#     1 - SETUP

In [0]:
from pyspark.sql import functions as F
import pandas as pd

CATALOG = 'portfolio_energia'
SCHEMA_SILVER = 'silver'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}')


# 2 - IDENTIFICANDO O FORMATO DAS TABELAS (COLUNAS, LINHAS DISTINTAS...)

2.1 Após análises e pesquisas externas, constatei que o período englobado pela tabela **ccee_pld_historico_semanal** não possui a mesma metodologia de cálculo que a tabela **ccee_pld_horario** e, por isso, aquela será desconsiderada para o estudo e não será tratada na camada **SILVER** deste projeto.

In [0]:
for tabela in ["ccee_pld_historico_semanal", "ccee_pld_horario", "ons_balanco_energia_subsistema"]:
    print(f"\n=== {tabela} ===")
    spark.sql(f"DESCRIBE {CATALOG}.bronze.{tabela}").show(truncate=False)

In [0]:
spark.sql(f"SELECT * FROM {CATALOG}.bronze.ccee_pld_horario LIMIT 5").show(truncate=False)
spark.sql(f"SELECT * FROM {CATALOG}.bronze.ons_balanco_energia_subsistema LIMIT 5").show(truncate=False)

# nesta etapa foi observado que existe uma diferença na nomenclatura de submercados que pode atrapalhar futuros joins
# por isso, foi alterada a lógica de criação das tables silver para que hada sempre o id_subsistema com o mesmo nome

In [0]:
# PLD horário
spark.sql(f"""
SELECT 
    COUNT(*) as total,
    COUNT(DISTINCT MES_REFERENCIA, SUBMERCADO, DIA, HORA) as distintos_com_mes
FROM {CATALOG}.bronze.ccee_pld_horario
""").show()

# Balanço energia
spark.sql(f"""
SELECT 
    COUNT(*) as total,
    COUNT(DISTINCT id_subsistema, din_instante) as distintos
FROM {CATALOG}.bronze.ons_balanco_energia_subsistema
""").show()

# 3 - Limpeza e padronização das tabelas

3.1 - Optei por um **overwrite** na camada silver pelo fato dessa ser a mesma estrutura de atualização que ocorre na Bronze.

In [0]:
df_silver_pld = (
    df_pld
    .withColumn(
        "instante",
        F.to_timestamp(
            F.concat(
                F.col("MES_REFERENCIA"),
                F.lpad(F.col("DIA"), 2, "0"),
                F.lpad(F.col("HORA"), 2, "0")
            ),
            "yyyyMMddHH"
        )
    )
    .withColumn( 
        "id_subsistema",
        F.when(F.trim(F.upper(F.col("SUBMERCADO"))) == "SUDESTE", "SE")
         .when(F.trim(F.upper(F.col("SUBMERCADO"))) == "SUL", "S")
         .when(F.trim(F.upper(F.col("SUBMERCADO"))) == "NORDESTE", "NE")
         .when(F.trim(F.upper(F.col("SUBMERCADO"))) == "NORTE", "N")
         .otherwise(None)
    )
    .withColumn("pld_hora", F.col("PLD_HORA").cast("decimal(10,2)"))
    .select(
        "instante",
        "id_subsistema",
        "pld_hora"
    )
    .dropDuplicates(["id_subsistema", "instante"])
)

(
    df_silver_pld.write
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(f"{CATALOG}.silver.pld_horario")
)
    

In [0]:
df_balanco = spark.table(f"{CATALOG}.bronze.ons_balanco_energia_subsistema")

colunas_val = [
    "val_gerhidraulica", "val_gertermica", "val_gereolica",
    "val_gersolar", "val_carga", "val_intercambio"
]

df_silver_balanco = (
    df_balanco
    .filter(F.trim(F.upper(F.col("id_subsistema"))) != "SIN")
    .withColumn("id_subsistema", F.trim(F.upper(F.col("id_subsistema"))))
    .withColumn("nom_subsistema", F.trim(F.upper(F.col("nom_subsistema"))))
    .withColumn("instante", F.col("din_instante").cast("timestamp"))
)

for col in colunas_val:
    df_silver_balanco = df_silver_balanco.withColumn(col, F.col(col).cast("decimal(15,4)"))

df_silver_balanco = (
    df_silver_balanco
    .select(
        "id_subsistema",
        "nom_subsistema",
        "instante",
        *colunas_val,
    )
    .dropDuplicates(["id_subsistema", "instante"])
)

(
    df_silver_balanco.write
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(f"{CATALOG}.silver.balanco_energia")
)

In [0]:
df_sin = (
    df_balanco
    .filter(F.trim(F.upper(F.col("id_subsistema"))) == "SIN")
    .withColumn("instante", F.col("din_instante").cast("timestamp"))
)

for col in colunas_val:
    df_sin = df_sin.withColumn(col, F.col(col).cast("decimal(15,4)"))

df_sin = (
    df_sin
    .select(
        "instante",
        *colunas_val
    )
    .dropDuplicates(["instante"])
)

(
    df_sin.write
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable(f"{CATALOG}.silver.balanco_energia_sin")

)

# 4 - Conferência da qualidade dos dados e se nenhum dado foi perdido nas transformações.

In [0]:
def comparar_contagem(bronze_tbl, silver_tbl, filtro_bronze="1=1"):
    total_bronze = spark.sql(f"SELECT COUNT(*) as c FROM {CATALOG}.bronze.{bronze_tbl} WHERE {filtro_bronze}").collect()[0]["c"]
    total_silver = spark.sql(f"SELECT COUNT(*) as c FROM {CATALOG}.silver.{silver_tbl}").collect()[0]["c"]
    print(f"{silver_tbl}: bronze={total_bronze} | silver={total_silver} | diferença={total_bronze - total_silver}")

comparar_contagem("ccee_pld_horario", "pld_horario")
comparar_contagem("ons_balanco_energia_subsistema", "balanco_energia", "TRIM(UPPER(id_subsistema)) != 'SIN'")
comparar_contagem("ons_balanco_energia_subsistema", "balanco_energia_sin", "TRIM(UPPER(id_subsistema)) = 'SIN'")

def checar_unicidade(tabela, colunas_chave):
    cols = ", ".join(colunas_chave)
    r = spark.sql(f"""
        SELECT COUNT(*) as total, COUNT(DISTINCT {cols}) as distintos
        FROM {CATALOG}.silver.{tabela}
    """).collect()[0]
    status = "OK" if r["total"] == r["distintos"] else "FALHA"
    print(f"{tabela}: total={r['total']} distintos={r['distintos']} -> {status}")

checar_unicidade("pld_horario", ["id_subsistema", "data_referencia"])
checar_unicidade("balanco_energia", ["id_subsistema", "instante"])
checar_unicidade("balanco_energia_sin", ["instante"])

def checar_nulos(tabela):
    df = spark.table(f"{CATALOG}.silver.{tabela}")
    df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(truncate=False)

checar_nulos("pld_horario")
checar_nulos("balanco_energia")
checar_nulos("balanco_energia_sin")

In [0]:
#verificando se há algum dia com menos de 24 horas; importante porque o pld é por hora
#não devem aparecer nenhuma linha aqui
spark.sql(f"""
    SELECT id_subsistema, DATE(data_referencia) as dia, COUNT(*) as horas_no_dia
    FROM {CATALOG}.silver.pld_horario
    GROUP BY id_subsistema, DATE(data_referencia)
    HAVING horas_no_dia != 24
    ORDER BY dia
""").show(20, truncate=False)

In [0]:
spark.sql(f"SELECT DISTINCT id_subsistema FROM {CATALOG}.silver.pld_horario").show()
spark.sql(f"SELECT DISTINCT id_subsistema, nom_subsistema FROM {CATALOG}.silver.balanco_energia").show()

In [0]:
spark.sql(f"SELECT * FROM {CATALOG}.silver.pld_horario LIMIT 5").show(truncate=False)
spark.sql(f"SELECT * FROM {CATALOG}.silver.balanco_energia LIMIT 5").show(truncate=False)
spark.sql(f"SELECT * FROM {CATALOG}.silver.balanco_energia_sin LIMIT 5").show(truncate=False)